## Exercice 3 : Créer un RAG : LlamaIndex + API + Persistance Storage + PrompTemplate 

In [1]:
# 0) Importer les modules
from dotenv import load_dotenv
import os
from llama_index.embeddings.azure_openai import AzureOpenAIEmbedding
from llama_index.llms.azure_openai import AzureOpenAI
from llama_index.core import (
    VectorStoreIndex, SimpleDirectoryReader,
    Settings, PromptTemplate,
    StorageContext, load_index_from_storage
)

In [2]:
# 1) Charger les variables d'environnements
load_dotenv()

llm_endpoint = os.getenv("LLM_ENDPOINT")
llm_api_key = os.getenv("LLM_API_KEY")
llm_api_version=os.getenv("LLM_API_VERSION")
llm_model = os.getenv("LLM_MODEL")

embed_endpoint = os.getenv("EMBED_ENDPOINT")
embed_api_key = os.getenv("EMBED_API_KEY")
embed_api_version = os.getenv("EMBED_API_VERSION")
embed_deploy = os.getenv("EMBED_DEPLOY")

In [3]:
# 2) Définir le modèle LLM
llm = AzureOpenAI(
    azure_endpoint=llm_endpoint,
    api_key=llm_api_key,
    deployment_name=llm_model,
    api_version=llm_api_version,
    temperature=0.2
)

In [4]:
# 3) Définir le modèle d'embedding
embed_model = AzureOpenAIEmbedding(
    azure_endpoint=embed_endpoint,
    api_key=embed_api_key,
    deployment_name=embed_deploy,
    api_version=embed_api_version
)

In [5]:
# 4) Définir les réglages globaux llamaindex
Settings.llm = llm
Settings.embed_model = embed_model
Settings.chunk_size = 512
Settings.chunk_overlap = 120

In [6]:
# 5) Ingestion + indexation persistante de la base documentaire

# Définir le dossier storage
storage_dir = "./storage"

# Si le dossier d'index existe déjà -> on le recharge
if os.path.exists(storage_dir) and os.listdir(storage_dir):
    print("Chargement de l'index existant depuis le dossier storage...")
    storage_context = StorageContext.from_defaults(persist_dir=storage_dir)
    index = load_index_from_storage(storage_context)
else:
    print("🆕 Aucun index trouvé, création à partir des documents...")
    docs = SimpleDirectoryReader("./data").load_data()
    index = VectorStoreIndex.from_documents(docs)
    # Sauvegarder le contexte
    index.storage_context.persist()
    print("Index créé et sauvegardé")

🆕 Aucun index trouvé, création à partir des documents...


2025-11-07 15:41:48,458 - INFO - HTTP Request: POST https://test-ai-agent-974-resource.cognitiveservices.azure.com/openai/deployments/text-embedding-3-large/embeddings?api-version=2024-02-01 "HTTP/1.1 200 OK"
2025-11-07 15:41:50,299 - INFO - HTTP Request: POST https://test-ai-agent-974-resource.cognitiveservices.azure.com/openai/deployments/text-embedding-3-large/embeddings?api-version=2024-02-01 "HTTP/1.1 200 OK"
2025-11-07 15:41:55,014 - INFO - HTTP Request: POST https://test-ai-agent-974-resource.cognitiveservices.azure.com/openai/deployments/text-embedding-3-large/embeddings?api-version=2024-02-01 "HTTP/1.1 200 OK"
2025-11-07 15:41:55,756 - INFO - HTTP Request: POST https://test-ai-agent-974-resource.cognitiveservices.azure.com/openai/deployments/text-embedding-3-large/embeddings?api-version=2024-02-01 "HTTP/1.1 200 OK"
2025-11-07 15:41:56,549 - INFO - HTTP Request: POST https://test-ai-agent-974-resource.cognitiveservices.azure.com/openai/deployments/text-embedding-3-large/embeddi

Index créé et sauvegardé


In [7]:
# 6) Créer un template Q&A
qa_tmpl = PromptTemplate(
    """Tu es un assistant concis. 
Contexte (extraits) :
{context_str}

Question utilisateur :
{query_str}

Consignes :
- Si le contexte contient la réponse, donne-la en 2-4 phrases claires.
- Si le contexte est insuffisant, propose une définition standard courte basée sur tes connaissances générales ET précise : 
  "(définition hors documents)". 
- Réponds en français, factuel, sans bavardage.
"""
)

In [ ]:
# 7) Effectuer une requête
qe = index.as_query_engine(
    similarity_top_k=8,
    # response_mode="compact",
    text_qa_template=qa_tmpl
)


In [9]:
qe.query("Quelle est la définition d'un développeur en intelligence artificielle ?")

2025-11-07 15:45:52,219 - INFO - HTTP Request: POST https://test-ai-agent-974-resource.cognitiveservices.azure.com/openai/deployments/text-embedding-3-large/embeddings?api-version=2024-02-01 "HTTP/1.1 200 OK"
2025-11-07 15:45:54,417 - INFO - HTTP Request: POST https://test-ai-agent-974-resource.cognitiveservices.azure.com/openai/deployments/gpt-4o-mini/chat/completions?api-version=2025-01-01-preview "HTTP/1.1 200 OK"
2025-11-07 15:45:55,852 - INFO - HTTP Request: POST https://test-ai-agent-974-resource.cognitiveservices.azure.com/openai/deployments/gpt-4o-mini/chat/completions?api-version=2025-01-01-preview "HTTP/1.1 200 OK"


Response(response="Un développeur en intelligence artificielle est un spécialiste du développement d'applicatifs informatiques autour de l'IA et de la Data Science. Son métier s'articule autour de la collecte, du stockage et de l'intégration de données, ainsi que de la conception d'applications intégrant des services d'intelligence artificielle. Il doit posséder des compétences en génie logiciel, interfaces Homme-Machine et technologies d'IA/Data Science.", source_nodes=[NodeWithScore(node=TextNode(id_='becb7000-4444-4fb6-8453-697dc5602902', embedding=None, metadata={'page_label': '4', 'file_name': '[2025 - Dev IA] Parcours de formation 3+16.pdf', 'file_path': 'c:\\Users\\olivi\\Desktop\\simplon\\module\\formateur\\phase2_dev_web_ia_ext\\ai_rag\\llamaindex\\corriges\\data\\[2025 - Dev IA] Parcours de formation 3+16.pdf', 'file_type': 'application/pdf', 'file_size': 2278472, 'creation_date': '2025-11-07', 'last_modified_date': '2025-10-10'}, excluded_embed_metadata_keys=['file_name', 'f

In [10]:
qe.query("Quelles sont les compétences en rapport avec les bases de données ?")

2025-11-07 15:46:56,472 - INFO - HTTP Request: POST https://test-ai-agent-974-resource.cognitiveservices.azure.com/openai/deployments/text-embedding-3-large/embeddings?api-version=2024-02-01 "HTTP/1.1 200 OK"
2025-11-07 15:46:59,135 - INFO - HTTP Request: POST https://test-ai-agent-974-resource.cognitiveservices.azure.com/openai/deployments/gpt-4o-mini/chat/completions?api-version=2025-01-01-preview "HTTP/1.1 200 OK"


Response(response="Les compétences en rapport avec les bases de données incluent la création de bases de données dans le respect du RGPD (C4), en élaborant des modèles conceptuels et physiques, ainsi que la programmation de leur import pour stocker les données du projet. De plus, le développement de requêtes SQL pour l'extraction de données depuis des systèmes de gestion de bases de données et des systèmes big data (C2) est également essentiel.", source_nodes=[NodeWithScore(node=TextNode(id_='9be4052d-fb67-4c7d-b254-2a502886c922', embedding=None, metadata={'page_label': '4', 'file_name': '[2025 - Dev IA] Parcours de formation 3+16.pdf', 'file_path': 'c:\\Users\\olivi\\Desktop\\simplon\\module\\formateur\\phase2_dev_web_ia_ext\\ai_rag\\llamaindex\\corriges\\data\\[2025 - Dev IA] Parcours de formation 3+16.pdf', 'file_type': 'application/pdf', 'file_size': 2278472, 'creation_date': '2025-11-07', 'last_modified_date': '2025-10-10'}, excluded_embed_metadata_keys=['file_name', 'file_type', 

In [11]:
# 8) Effectuer une requête en mode chat
qc = index.as_chat_engine(
    similarity_top_k=8,
    response_mode="compact",
    text_qa_template=qa_tmpl
)
response = qc.chat("Quelle est la définition d'un développeur en intelligence artificielle ?")
print(response)
response = qc.chat("En quoi consiste la phase 0 ?")
print(response)


2025-11-06 20:44:21,849 - INFO - Condensed question: Quelle est la définition d'un développeur en intelligence artificielle ?
2025-11-06 20:44:23,348 - INFO - HTTP Request: POST https://test-ai-agent-974-resource.cognitiveservices.azure.com/openai/deployments/text-embedding-3-large/embeddings?api-version=2024-02-01 "HTTP/1.1 200 OK"
2025-11-06 20:44:28,315 - INFO - HTTP Request: POST https://test-ai-agent-974-resource.cognitiveservices.azure.com/openai/deployments/gpt-4o-mini/chat/completions?api-version=2025-01-01-preview "HTTP/1.1 200 OK"
2025-11-06 20:44:32,001 - INFO - HTTP Request: POST https://test-ai-agent-974-resource.cognitiveservices.azure.com/openai/deployments/gpt-4o-mini/chat/completions?api-version=2025-01-01-preview "HTTP/1.1 200 OK"


Un développeur en intelligence artificielle est un spécialiste du développement d'applicatifs informatiques autour de l'IA et de la Data Science. Ce métier s'articule autour de trois activités principales :

1. **Collecte, stockage et mise à disposition des données** d’un projet en intelligence artificielle.
2. **Intégration de modèles et de services d’intelligence artificielle**.
3. **Réalisation d'applications intégrant un service d’intelligence artificielle**.

Pour accomplir ces activités, le développeur en IA doit posséder plusieurs compétences, telles que l'automatisation de l'extraction de données, le développement de requêtes SQL, la création d'APIs, et la mise en place de tests automatisés. Il est également impliqué dans des tâches comme le monitoring des modèles d'IA et la coordination de la réalisation technique d'applications en utilisant des méthodes agiles et des approches MLOps.

En résumé, le développeur en intelligence artificielle joue un rôle clé dans la démocratisat

2025-11-06 20:44:32,718 - INFO - HTTP Request: POST https://test-ai-agent-974-resource.cognitiveservices.azure.com/openai/deployments/gpt-4o-mini/chat/completions?api-version=2025-01-01-preview "HTTP/1.1 200 OK"
2025-11-06 20:44:32,722 - INFO - Condensed question: Qu'est-ce que la phase 0 dans le développement d'un projet en intelligence artificielle ?
2025-11-06 20:44:34,152 - INFO - HTTP Request: POST https://test-ai-agent-974-resource.cognitiveservices.azure.com/openai/deployments/text-embedding-3-large/embeddings?api-version=2024-02-01 "HTTP/1.1 200 OK"
2025-11-06 20:44:38,965 - INFO - HTTP Request: POST https://test-ai-agent-974-resource.cognitiveservices.azure.com/openai/deployments/gpt-4o-mini/chat/completions?api-version=2025-01-01-preview "HTTP/1.1 200 OK"
2025-11-06 20:44:41,526 - INFO - HTTP Request: POST https://test-ai-agent-974-resource.cognitiveservices.azure.com/openai/deployments/gpt-4o-mini/chat/completions?api-version=2025-01-01-preview "HTTP/1.1 200 OK"


La phase 0, intitulée "La prairie", consiste à reproduire des interfaces et des traitements de données. Elle dure 70 heures et comprend plusieurs activités pratiques. Voici les principales tâches à réaliser durant cette phase :

1. **Développer la nouvelle version d’une application de campagne marketing** : Cela implique de mettre à jour ou de créer une application qui gère des campagnes marketing, en intégrant éventuellement des fonctionnalités nouvelles ou améliorées.

2. **Analyser les résultats d'un questionnaire de satisfaction par des requêtes SQL** : Cette tâche consiste à utiliser des requêtes SQL pour extraire et analyser des données issues d'un questionnaire de satisfaction, permettant ainsi d'évaluer la performance ou la satisfaction des utilisateurs.

Cette phase est essentielle pour poser les bases du développement d'applications d'intelligence artificielle en se concentrant sur des compétences fondamentales en développement et en gestion de données. Elle permet aux partic

In [11]:
# Annexe : Inspecter ce que LlamaIndex injecte
# 1) Faire la requête comme d’habitude
resp = qe.query("Quelle est la définition d'un développeur en intelligence artificielle ?")

# 2) Le texte final renvoyé par le LLM
print("Réponse du modèle:\n", resp.response)

# 3) Les passages (source nodes) qui ont servi de contexte
for i, sn in enumerate(resp.source_nodes):
    print(f"\n--- Source #{i+1} | score={sn.score:.3f}")
    print("Fichier:", sn.node.metadata.get("file_name"))
    # le contenu exact passé au LLM (chunk)
    print(sn.node.get_content()[:700], "...")


2025-11-07 15:52:37,463 - INFO - HTTP Request: POST https://test-ai-agent-974-resource.cognitiveservices.azure.com/openai/deployments/text-embedding-3-large/embeddings?api-version=2024-02-01 "HTTP/1.1 200 OK"
2025-11-07 15:52:40,260 - INFO - HTTP Request: POST https://test-ai-agent-974-resource.cognitiveservices.azure.com/openai/deployments/gpt-4o-mini/chat/completions?api-version=2025-01-01-preview "HTTP/1.1 200 OK"
2025-11-07 15:52:41,649 - INFO - HTTP Request: POST https://test-ai-agent-974-resource.cognitiveservices.azure.com/openai/deployments/gpt-4o-mini/chat/completions?api-version=2025-01-01-preview "HTTP/1.1 200 OK"


Réponse du modèle:
 Un développeur en intelligence artificielle est un spécialiste du développement d'applicatifs informatiques liés à l'IA et à la Data Science. Il réalise des activités telles que la collecte et le stockage des données, l'intégration de modèles d'IA, et la création d'applications exploitant ces services. Ce métier nécessite des compétences en automatisation, développement d'API, et mise en place de tests et de monitoring.

--- Source #1 | score=0.584
Fichier: [2025 - Dev IA] Parcours de formation 3+16.pdf
Il
 
est
 
donc
 
un
 
spécialiste
 
du
 
développement
 
informatique,
 
du
 
génie
 
logiciel
 
et
 
des
 
interfaces
 
Homme-Machine,
 
avec
 
une
 
très
 
bonne
 
connaissance
 
des
 
technologies
 
d'IA/Data
 
Science.
 
Le  métier  de  développeur·se  en  intelligence  artiﬁcielle  s’articule  alors  autour  de  trois  activités  
principales
 
:
 
●  A1.  Réaliser  la  collecte,  le  stockage  et  la  mise  à  disposition  des  données  d’un  projet  en  
inte